# 🚀 Full Project Source Code
This notebook contains the complete source code for all project scripts:
1. `generate_data.py`
2. `train_model.py`
3. `segmentor.py`
4. `app.py`

## Data Generation Script (`generate_data.py`)

In [1]:
"""
generate_data.py
Generates a synthetic customer dataset for segmentation analysis.
"""
import numpy as np
import pandas as pd
import os

np.random.seed(42)
N_CUSTOMERS = 2000

def generate_archetype(n: int, name: str, params: dict) -> pd.DataFrame:
    """
    Generates a DataFrame for a specific customer archetype based on provided parameters.
    
    Args:
        n (int): Number of customers to generate.
        name (str): Label for the archetype (e.g., 'A').
        params (dict): Dictionary defining bounds/means for features.
        
    Returns:
        pd.DataFrame: Generated customer data.
    """
    df = pd.DataFrame()
    df['archetype'] = [name] * n
    
    # Base features (same for all unless specified)
    df['age'] = np.clip(np.random.normal(42, 13, n), 18, 75).astype(int)
    df['gender'] = np.random.choice(['F', 'M'], n, p=[0.55, 0.45])
    
    categories = ['Electronics', 'Clothing', 'Grocery', 'Home', 'Beauty', 'Sports', 'Books']
    df['product_category_preference'] = np.random.choice(categories, n)
    df['support_tickets'] = np.clip(np.random.normal(2, 2, n), 0, 20).astype(int)
    
    # Generate continuous features with noise
    continuous_features = [
        'annual_income', 'spending_score', 'recency_days', 'frequency', 
        'monetary', 'discount_usage_rate', 'loyalty_years', 'returns_rate', 
        'online_purchase_ratio'
    ]
    
    for feat in continuous_features:
        if feat in params:
            low, high = params[feat]
            # using uniform distribution as base
            base_values = np.random.uniform(low, high, n)
            # Add Gaussian noise (5% of range)
            noise_std = (high - low) * 0.05
            noisy_values = base_values + np.random.normal(0, noise_std, n)
            
            # Clip back to realistic absolute ranges to ensure valid values
            clip_low = max(0, low - noise_std * 2) 
            clip_high = high + noise_std * 2
            
            # Specific absolute bounds to enforce sanity
            if feat == 'spending_score': clip_low, clip_high = 1, 100
            if feat == 'recency_days': clip_low, clip_high = 1, 365
            if feat == 'discount_usage_rate': clip_low, clip_high = 0.0, 1.0
            if feat == 'returns_rate': clip_low, clip_high = 0.0, 1.0
            if feat == 'online_purchase_ratio': clip_low, clip_high = 0.0, 1.0
            if feat == 'frequency': clip_low = 1
            if feat == 'monetary': clip_low = 50
            if feat == 'loyalty_years': clip_low, clip_high = 0, 15
            
            df[feat] = np.clip(noisy_values, clip_low, clip_high)
        else:
            # Default fallback if not defined in archetype params
            if feat == 'online_purchase_ratio':
                df[feat] = np.clip(np.random.uniform(0.1, 0.9, n) + np.random.normal(0, 0.05, n), 0, 1)
            else:
                df[feat] = 0
                
    # Formatting / Rounding where appropriate
    df['annual_income'] = df['annual_income'].round(2)
    df['spending_score'] = df['spending_score'].astype(int)
    df['recency_days'] = df['recency_days'].astype(int)
    df['frequency'] = df['frequency'].astype(int)
    df['monetary'] = df['monetary'].round(2)
    df['loyalty_years'] = df['loyalty_years'].astype(int)
    
    return df

def generate_data() -> None:
    """
    Generates the entire dataset by combining archetypes and deriving new features.
    """
    print("Generating archetypes...")
    
    # Archetype Definitions
    archetypes_def = {
        'A': { # Premium Loyalists
            'n': 400,
            'annual_income': (80000, 200000),
            'spending_score': (75, 100),
            'recency_days': (1, 30),
            'frequency': (30, 80),
            'monetary': (3000, 15000),
            'discount_usage_rate': (0.0, 0.1),
            'loyalty_years': (5, 15),
            'returns_rate': (0.0, 0.05),
            'online_purchase_ratio': (0.4, 0.8) # Sensible default
        },
        'B': { # Occasional Shoppers
            'n': 500,
            'annual_income': (30000, 60000),
            'spending_score': (35, 65),
            'recency_days': (60, 180),
            'frequency': (3, 15),
            'monetary': (200, 1200),
            'discount_usage_rate': (0.2, 0.5),
            'loyalty_years': (0, 3),
            'returns_rate': (0.05, 0.15),
            'online_purchase_ratio': (0.2, 0.6)
        },
        'C': { # Bargain Hunters
            'n': 450,
            'annual_income': (20000, 50000),
            'spending_score': (40, 70),
            'recency_days': (1, 45),
            'frequency': (20, 60),
            'monetary': (500, 3000),
            'discount_usage_rate': (0.6, 1.0),
            'loyalty_years': (2, 8),
            'returns_rate': (0.1, 0.3),
            'online_purchase_ratio': (0.4, 0.9)
        },
        'D': { # At-Risk High-Value
            'n': 350,
            'annual_income': (70000, 150000),
            'spending_score': (10, 35),
            'recency_days': (180, 365),
            'frequency': (1, 6),
            'monetary': (1500, 8000),
            'discount_usage_rate': (0.0, 0.2),
            'loyalty_years': (3, 12),
            'returns_rate': (0.15, 0.4),
            'online_purchase_ratio': (0.3, 0.7)
        },
        'E': { # Young Explorers
            'n': 300,
            'annual_income': (15000, 35000),
            'spending_score': (60, 95),
            'recency_days': (1, 60),
            'frequency': (8, 25),
            'monetary': (100, 800),
            'discount_usage_rate': (0.3, 0.8),
            'loyalty_years': (0, 2),
            'returns_rate': (0.0, 0.1),
            'online_purchase_ratio': (0.7, 1.0)
        }
    }
    
    dfs = []
    archetype_counts = {}
    
    for name, a_def in archetypes_def.items():
        n = a_def['n']
        archetype_counts[name] = n
        params = {k: v for k, v in a_def.items() if k != 'n'}
        dfs.append(generate_archetype(n, name, params))
        
    df = pd.concat(dfs, ignore_index=True)
    
    # Shuffle dataset
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Assign customer_id formatted as CUST_0001
    df.insert(0, 'customer_id', [f"CUST_{i+1:04d}" for i in range(len(df))])
    
    # STEP 4: Add derived features
    # avg_order_value: monetary / frequency (clip to min 1.0)
    df['avg_order_value'] = np.clip(df['monetary'] / df['frequency'], 1.0, None).round(2)
    
    # clv_score: (frequency * monetary) / (recency_days + 1)
    df['clv_score'] = ((df['frequency'] * df['monetary']) / (df['recency_days'] + 1)).round(2)
    
    # engagement_score: (1/recency_days * 100) + (frequency * 2) + (online_purchase_ratio * 20) - (support_tickets * 3)
    eng_score = (1 / df['recency_days'] * 100) + (df['frequency'] * 2) + \
                (df['online_purchase_ratio'] * 20) - (df['support_tickets'] * 3)
    # clip to 0-200
    df['engagement_score'] = np.clip(eng_score, 0, 200).round(2)
    
    # STEP 5: Shuffle and save
    os.makedirs("data", exist_ok=True)
    output_path = "data/customers.csv"
    df.to_csv(output_path, index=False)
    
    print("Customer dataset generated:")
    print(f"  Total customers: {N_CUSTOMERS}")
    print(f"  Archetypes: A={archetype_counts['A']}, B={archetype_counts['B']}, C={archetype_counts['C']}, D={archetype_counts['D']}, E={archetype_counts['E']}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Avg annual income: ${df['annual_income'].mean():,.0f}")
    print(f"  Avg monetary: ${df['monetary'].mean():,.0f}")
    print(f"Data saved to {output_path} ✅")

if __name__ == "__main__":
    try:
        generate_data()
    except Exception as e:
        print(f"Error generating data: {e}")


Generating archetypes...
Customer dataset generated:
  Total customers: 2000
  Archetypes: A=400, B=500, C=450, D=350, E=300
  Columns: ['customer_id', 'archetype', 'age', 'gender', 'product_category_preference', 'support_tickets', 'annual_income', 'spending_score', 'recency_days', 'frequency', 'monetary', 'discount_usage_rate', 'loyalty_years', 'returns_rate', 'online_purchase_ratio', 'avg_order_value', 'clv_score', 'engagement_score']
  Avg annual income: $69,898
  Avg monetary: $3,257
Data saved to data/customers.csv ✅


## Model Training Pipeline (`train_model.py`)

In [2]:
"""
train_model.py
Trains K-Means and PCA models, finds optimal K, and saves artifacts.
"""
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# STEP 1 — Define features for clustering
CLUSTER_FEATURES = [
    'annual_income',       # Economic power
    'spending_score',      # Spending willingness
    'recency_days',        # How recently they bought
    'frequency',           # How often they buy
    'monetary',            # How much they spend
    'avg_order_value',     # Basket size
    'online_purchase_ratio', # Digital preference
    'loyalty_years',       # Relationship length
    'discount_usage_rate', # Price sensitivity
    'returns_rate',        # Satisfaction proxy
    'support_tickets',     # Service usage
    'clv_score',           # CLV proxy
    'engagement_score',    # Overall engagement
]

def main() -> None:
    print("Loading data...")
    # STEP 2 — Load and scale
    try:
        df = pd.read_csv("data/customers.csv")
    except FileNotFoundError:
        print("Error: data/customers.csv not found. Run generate_data.py first.")
        return

    X = df[CLUSTER_FEATURES].copy()
    
    # Check for NaN — fill with column median if any exist
    if X.isnull().sum().sum() > 0:
        X = X.fillna(X.median())
        
    print(f"Feature matrix shape: {X.shape}")
    print("\nFeature stats:")
    print(X.describe().loc[['min', 'max', 'mean']].round(2))
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(scaler, "models/scaler.pkl")

    # STEP 3 — PCA for visualization (2 components)
    pca_2d = PCA(n_components=2, random_state=42)
    X_pca_2d = pca_2d.fit_transform(X_scaled)
    joblib.dump(pca_2d, "models/pca_model.pkl")
    print(f"\nPCA explained variance: {pca_2d.explained_variance_ratio_}")
    print(f"Total variance captured: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%")
    
    # Compute PCA with n_components=min(13, n_samples)
    pca_full = PCA(n_components=min(len(CLUSTER_FEATURES), len(X)), random_state=42)
    pca_full.fit(X_scaled)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    print("Cumulative explained variance:")
    for i, cum_var in enumerate(cumulative_variance):
        print(f"  {i+1} components: {cum_var*100:.1f}%")
        
    # STEP 4 — Find Optimal K using Elbow Method + Silhouette Score
    k_range = range(2, 11)
    inertias, silhouette_scores, db_scores = [], [], []

    print("\nTesting K values from 2 to 10...")
    print(f"{'K':>3} | {'Inertia':>12} | {'Silhouette':>12} | {'Davies-Bouldin':>15}")
    for k in k_range:
        km = KMeans(n_clusters=k, init='k-means++', n_init=20,
                    max_iter=500, random_state=42)
        labels = km.fit_predict(X_scaled)
        inertias.append(km.inertia_)
        silhouette_scores.append(silhouette_score(X_scaled, labels))
        db_scores.append(davies_bouldin_score(X_scaled, labels))
        print(f"{k:>3} | {km.inertia_:>12,.0f} | {silhouette_scores[-1]:>12.4f} | {db_scores[-1]:>15.4f}")

    best_k_silhouette = k_range[np.argmax(silhouette_scores)]
    print(f"\nSilhouette recommends K={best_k_silhouette}")
    print("Selected K=5 to match known archetype count ✅")
    
    # STEP 5 — Train final K-Means with K=5
    OPTIMAL_K = 5
    kmeans = KMeans(
        n_clusters=OPTIMAL_K,
        init='k-means++',     # Smart centroid initialization
        n_init=50,            # Run 50 times, keep best result
        max_iter=1000,        # Maximum iterations per run
        tol=1e-6,             # Convergence tolerance
        random_state=42
    )
    cluster_labels = kmeans.fit_predict(X_scaled)
    joblib.dump(kmeans, "models/kmeans_model.pkl")

    # STEP 6 — Evaluate final model
    final_sil = silhouette_score(X_scaled, cluster_labels)
    final_db = davies_bouldin_score(X_scaled, cluster_labels)
    print(f"\nFinal K-Means: Inertia={kmeans.inertia_:,.0f} | Silhouette={final_sil:.3f} | DB={final_db:.3f}")
    
    print("\nCluster sizes:")
    unique, counts = np.unique(cluster_labels, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"  Cluster {u}: {c}")

    # STEP 7 — Build segment profiles
    df['cluster'] = cluster_labels
    cluster_profiles = df.groupby('cluster')[CLUSTER_FEATURES].mean()
    
    # Assign names dynamically based on distinct feature characteristics
    cluster_monetary = cluster_profiles['monetary'].sort_values(ascending=False).index.tolist()
    premium_cluster = cluster_monetary[0]
    
    cluster_recency = cluster_profiles['recency_days'].sort_values(ascending=False).index.tolist()
    at_risk_cluster = next(c for c in cluster_recency if c != premium_cluster)
    
    cluster_discount = cluster_profiles['discount_usage_rate'].sort_values(ascending=False).index.tolist()
    bargain_cluster = next(c for c in cluster_discount if c not in [premium_cluster, at_risk_cluster])
    
    cluster_online = cluster_profiles['online_purchase_ratio'].sort_values(ascending=False).index.tolist()
    young_cluster = next(c for c in cluster_online if c not in [premium_cluster, at_risk_cluster, bargain_cluster])
    
    occasional_cluster = next(c for c in range(5) if c not in [premium_cluster, at_risk_cluster, bargain_cluster, young_cluster])
    
    SEGMENT_NAMES = {
        premium_cluster: "Premium Loyalists",
        at_risk_cluster: "At-Risk High-Value",
        bargain_cluster: "Bargain Hunters",
        young_cluster: "Young Explorers",
        occasional_cluster: "Occasional Shoppers"
    }
    
    print("\nAssigned Segment Names:")
    for c, name in SEGMENT_NAMES.items():
        print(f"  Cluster {c}: {name}")
    
    profiles = {}
    for c in range(OPTIMAL_K):
        subset = df[df['cluster'] == c]
        profile = {
            'size': len(subset),
            'pct': len(subset) / len(df),
            'dominant_product_category': subset['product_category_preference'].mode()[0],
            'dominant_gender': subset['gender'].mode()[0],
            'avg_age': subset['age'].mean(),
            'name': SEGMENT_NAMES[c]
        }
        for feat in CLUSTER_FEATURES:
            profile[f'{feat}_mean'] = subset[feat].mean()
            
        profile['annual_income_median'] = subset['annual_income'].median()
        profile['monetary_median'] = subset['monetary'].median()
        profile['frequency_median'] = subset['frequency'].median()
        
        profiles[c] = profile

    joblib.dump(profiles, "models/segment_profiles.pkl")
    
    # STEP 8 — Add cluster labels and PCA coords to dataset
    df['segment_name'] = df['cluster'].map(SEGMENT_NAMES)
    df['pca_x'] = X_pca_2d[:, 0]
    df['pca_y'] = X_pca_2d[:, 1]
    df.to_csv("data/customers_segmented.csv", index=False)
    
    # STEP 9 — Print full summary
    print("\nAll models saved ✅")
    
    # STEP 10 — Compare with DBSCAN
    dbscan = DBSCAN(eps=0.8, min_samples=10, n_jobs=-1)
    db_labels = dbscan.fit_predict(X_scaled)
    n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = list(db_labels).count(-1)
    print(f"\nDBSCAN comparison: {n_clusters_db} clusters, {n_noise} noise points")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Error during training: {e}")


Loading data...
Feature matrix shape: (2000, 13)

Feature stats:
      annual_income  spending_score  recency_days  frequency  monetary  \
min        13811.59             8.0          1.00       1.00     50.00   
max       206829.32           100.0        365.00      84.00  15690.47   
mean       69897.82            57.3         89.75      24.96   3257.16   

      avg_order_value  online_purchase_ratio  loyalty_years  \
min              2.47                   0.17           0.00   
max           8198.49                   1.00          15.00   
mean           465.87                   0.58           4.49   

      discount_usage_rate  returns_rate  support_tickets  clv_score  \
min                  0.00          0.00             0.00       2.84   
max                  1.00          0.42             8.00  518613.86   
mean                 0.38          0.13             1.82   10826.36   

      engagement_score  
min               0.00  
max             200.00  
mean             62.55  


## ML Brain and Chart Engine (`segmentor.py`)

In [3]:
import os
import joblib
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import silhouette_samples
from sklearn.cluster import DBSCAN, KMeans

# CONSTANTS
CLUSTER_FEATURES = [
    'annual_income', 'spending_score', 'recency_days', 'frequency',
    'monetary', 'avg_order_value', 'online_purchase_ratio',
    'loyalty_years', 'discount_usage_rate', 'returns_rate',
    'support_tickets', 'clv_score', 'engagement_score'
]

SEGMENT_NAMES = {
    0: "Premium Loyalists",
    1: "Occasional Shoppers",
    2: "Bargain Hunters",
    3: "At-Risk High-Value",
    4: "Young Explorers"
}

SEGMENT_COLORS = {
    "Premium Loyalists":    "#F39C12",
    "Occasional Shoppers":  "#3498DB",
    "Bargain Hunters":      "#2ECC71",
    "At-Risk High-Value":   "#E74C3C",
    "Young Explorers":      "#9B59B6"
}

SEGMENT_EMOJIS = {
    "Premium Loyalists":    "👑",
    "Occasional Shoppers":  "🛍️",
    "Bargain Hunters":      "🏷️",
    "At-Risk High-Value":   "⚠️",
    "Young Explorers":      "🚀"
}

SEGMENT_STRATEGIES = {
    "Premium Loyalists":
        "VIP program, exclusive early access, personal account manager, "
        "luxury product recommendations, anniversary rewards",
    "Occasional Shoppers":
        "Re-engagement campaigns, seasonal promotions, browse abandonment "
        "emails, convenience-focused messaging, free shipping offers",
    "Bargain Hunters":
        "Flash sale alerts, loyalty points program, bulk buy discounts, "
        "clearance section highlights, referral rewards",
    "At-Risk High-Value":
        "Win-back campaign, personal outreach call, premium upgrade offer, "
        "survey to identify pain points, exclusive return offer",
    "Young Explorers":
        "Social media engagement, trend-first notifications, student "
        "discounts, gamification, influencer collaborations"
}

FEATURE_DISPLAY = {
    'annual_income':          ('Annual Income ($)',       '$,.0f'),
    'spending_score':         ('Spending Score (1-100)',  '.1f'),
    'recency_days':           ('Days Since Last Purchase', '.0f'),
    'frequency':              ('Purchase Frequency/Year', '.1f'),
    'monetary':               ('Annual Spend ($)',         '$,.0f'),
    'avg_order_value':        ('Avg Order Value ($)',      '$,.2f'),
    'online_purchase_ratio':  ('Online Purchase Ratio',   '.1%'),
    'loyalty_years':          ('Loyalty Years',           '.1f'),
    'discount_usage_rate':    ('Discount Usage Rate',     '.1%'),
    'returns_rate':           ('Returns Rate',            '.1%'),
    'support_tickets':        ('Support Tickets/Year',    '.1f'),
    'clv_score':              ('CLV Score (proxy)',        ',.1f'),
    'engagement_score':       ('Engagement Score',        '.1f'),
}

# MODEL LOADING
try:
    kmeans = joblib.load("models/kmeans_model.pkl")
    pca = joblib.load("models/pca_model.pkl")
    scaler = joblib.load("models/scaler.pkl")
    profiles = joblib.load("models/segment_profiles.pkl")
except FileNotFoundError:
    pass # Expected during initial definition or before running train_model.py

# FUNCTIONS

def load_segmented_data() -> pd.DataFrame:
    """Loads pre-segmented customer data."""
    try:
        df = pd.read_csv("data/customers_segmented.csv")
        if 'segment_name' not in df.columns and 'cluster' in df.columns:
            df['segment_name'] = df['cluster'].map(SEGMENT_NAMES)
        return df
    except FileNotFoundError:
        raise FileNotFoundError("Segmented data not found. Run train_model.py first.")

def assign_segments(df_new: pd.DataFrame) -> pd.DataFrame:
    """Predicts segments for new raw customer data."""
    df = df_new.copy()
    
    # Engineer features if not present
    if 'clv_score' not in df.columns:
        df['clv_score'] = ((df['frequency'] * df['monetary']) / (df['recency_days'] + 1)).round(2)
    if 'engagement_score' not in df.columns:
        eng_score = (1 / df['recency_days'] * 100) + (df['frequency'] * 2) + \
                    (df.get('online_purchase_ratio', 0) * 20) - (df.get('support_tickets', 0) * 3)
        df['engagement_score'] = np.clip(eng_score, 0, 200).round(2)
    if 'avg_order_value' not in df.columns:
        df['avg_order_value'] = np.clip(df['monetary'] / df['frequency'], 1.0, None).round(2)
        
    # Align to CLUSTER_FEATURES
    for col in CLUSTER_FEATURES:
        if col not in df.columns:
            df[col] = 0
            
    X = df[CLUSTER_FEATURES].copy()
    
    # Scale and predict
    X_scaled = scaler.transform(X)
    cluster_labels = kmeans.predict(X_scaled)
    
    # Add cluster and segment name
    df['cluster'] = cluster_labels
    df['segment_name'] = df['cluster'].map(SEGMENT_NAMES)
    
    # Add PCA coordinates
    X_pca = pca.transform(X_scaled)
    df['pca_x'] = X_pca[:, 0]
    df['pca_y'] = X_pca[:, 1]
    
    return df

def get_segment_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Returns a summary of each segment."""
    summary = []
    total_customers = len(df)
    
    for name in SEGMENT_NAMES.values():
        subset = df[df['segment_name'] == name]
        if len(subset) == 0:
            continue
            
        summary.append({
            'segment_name': name,
            'emoji': SEGMENT_EMOJIS.get(name, "🔘"),
            'count': len(subset),
            'pct_of_total': f"{(len(subset) / total_customers * 100):.1f}%",
            'avg_income': subset['annual_income'].mean(),
            'avg_monetary': subset['monetary'].mean(),
            'avg_frequency': subset['frequency'].mean(),
            'avg_recency': subset['recency_days'].mean(),
            'avg_spending_score': subset['spending_score'].mean(),
            'avg_loyalty_years': subset['loyalty_years'].mean(),
            'color': SEGMENT_COLORS.get(name, "#FFFFFF"),
            'strategy': SEGMENT_STRATEGIES.get(name, "")
        })
        
    return pd.DataFrame(summary)

def get_pca_scatter(df: pd.DataFrame, highlight_customer: str = None) -> go.Figure:
    fig = px.scatter(
        df, x='pca_x', y='pca_y', color='segment_name',
        color_discrete_map=SEGMENT_COLORS,
        hover_data=['customer_id', 'segment_name', 'annual_income', 'monetary', 'spending_score', 'frequency'],
        title="Customer Segments — PCA Visualization (2D)",
        labels={'pca_x': "PC1 (Economic Power + Monetary)", 'pca_y': "PC2 (Engagement + Recency)"},
        opacity=0.7, template='plotly_dark'
    )
    fig.update_traces(marker=dict(size=5))
    
    # Add centroids
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    fig.add_trace(go.Scatter(
        x=centroids_pca[:, 0], y=centroids_pca[:, 1],
        mode='markers', marker=dict(size=20, symbol='star', color='white', line=dict(width=1, color='black')),
        name='Centroids', hoverinfo='skip'
    ))
    
    if highlight_customer and 'customer_id' in df.columns:
        cust = df[df['customer_id'] == highlight_customer]
        if not cust.empty:
            fig.add_trace(go.Scatter(
                x=cust['pca_x'], y=cust['pca_y'],
                mode='markers', marker=dict(size=15, color='white', line=dict(width=2, color='black')),
                name=f"Customer: {highlight_customer}"
            ))
            fig.add_annotation(
                x=cust['pca_x'].iloc[0], y=cust['pca_y'].iloc[0],
                text=highlight_customer, showarrow=True, arrowhead=1
            )
            
    return fig

def get_radar_chart(segment_name: str) -> go.Figure:
    features = [
        'spending_score', 'frequency', 'monetary', 'loyalty_years',
        'online_purchase_ratio', 'engagement_score', 'recency_days', 'discount_usage_rate'
    ]
    
    global_df = load_segmented_data()
    segment_df = global_df[global_df['segment_name'] == segment_name]
    
    if segment_df.empty:
        return go.Figure()
        
    global_means = []
    segment_means = []
    
    for f in features:
        min_v = global_df[f].min()
        max_v = global_df[f].max()
        range_v = max_v - min_v if max_v != min_v else 1
        
        glob_m = (global_df[f].mean() - min_v) / range_v
        seg_m = (segment_df[f].mean() - min_v) / range_v
        
        if f == 'recency_days':
            glob_m = 1 - glob_m
            seg_m = 1 - seg_m
            
        global_means.append(glob_m)
        segment_means.append(seg_m)
        
    features.append(features[0])
    global_means.append(global_means[0])
    segment_means.append(segment_means[0])
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatterpolar(
        r=global_means, theta=features, fill=None,
        mode='lines', line=dict(color='gray', dash='dash'), name='Global Average'
    ))
    
    fig.add_trace(go.Scatterpolar(
        r=segment_means, theta=features, fill='toself',
        line=dict(color=SEGMENT_COLORS.get(segment_name, 'white')),
        name=segment_name
    ))
    
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title=f"{segment_name} — Feature Profile",
        template='plotly_dark', showlegend=True
    )
    return fig

def get_segment_comparison_bar(df: pd.DataFrame, feature: str) -> go.Figure:
    grouped = df.groupby('segment_name')[feature].agg(['mean', 'std']).reset_index()
    grouped = grouped.sort_values(by='mean', ascending=False)
    
    disp_name = FEATURE_DISPLAY.get(feature, (feature, ''))[0]
    
    fig = px.bar(
        grouped, x='segment_name', y='mean', error_y='std',
        color='segment_name', color_discrete_map=SEGMENT_COLORS,
        title=f"Segment Comparison — {disp_name}",
        template='plotly_dark'
    )
    fig.update_layout(xaxis_title="Segment", yaxis_title=disp_name)
    return fig

def get_rfm_3d_scatter(df: pd.DataFrame) -> go.Figure:
    df_plot = df.copy()
    min_i = df_plot['annual_income'].min()
    max_i = df_plot['annual_income'].max()
    # Handle division by zero
    range_i = max_i - min_i if max_i > min_i else 1
    df_plot['income_size'] = ((df_plot['annual_income'] - min_i) / range_i) * 20 + 5
    
    fig = px.scatter_3d(
        df_plot, x='recency_days', y='frequency', z='monetary',
        color='segment_name', color_discrete_map=SEGMENT_COLORS,
        size='income_size',
        hover_data=['customer_id', 'segment_name', 'recency_days', 'frequency', 'monetary'],
        title="RFM 3D Customer Map",
        template='plotly_dark'
    )
    return fig

def get_income_spend_scatter(df: pd.DataFrame) -> go.Figure:
    fig = px.scatter(
        df, x='annual_income', y='spending_score', color='segment_name',
        color_discrete_map=SEGMENT_COLORS, marginal_x='histogram', marginal_y='histogram',
        hover_data=['customer_id', 'segment_name', 'monetary', 'frequency'],
        title="Annual Income vs Spending Score by Segment",
        template='plotly_dark'
    )
    return fig

def get_feature_distribution_box(df: pd.DataFrame, feature: str) -> go.Figure:
    disp_name = FEATURE_DISPLAY.get(feature, (feature, ''))[0]
    fig = px.box(
        df, x='segment_name', y=feature, color='segment_name',
        color_discrete_map=SEGMENT_COLORS, points='outliers',
        title=f"Distribution of {disp_name}",
        template='plotly_dark'
    )
    return fig

def get_segment_heatmap(df: pd.DataFrame) -> go.Figure:
    means = df.groupby('segment_name')[CLUSTER_FEATURES].mean()
    normalized = (means - means.min()) / (means.max() - means.min() + 1e-9)
    
    fig = px.imshow(
        normalized, color_continuous_scale='RdYlGn',
        aspect='auto',
        title="Segment Feature Profile Heatmap (Normalized)",
        template='plotly_dark'
    )
    fig.update_traces(text=means.round(1).values, texttemplate="%{text}")
    return fig

def get_elbow_chart(inertias: list, silhouettes: list) -> go.Figure:
    k_range = list(range(2, 2 + len(inertias)))
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=k_range, y=inertias, name='Inertia', line=dict(color='blue'), yaxis='y1'))
    fig.add_trace(go.Scatter(x=k_range, y=silhouettes, name='Silhouette', line=dict(color='green'), yaxis='y2'))
    
    fig.update_layout(
        title="Elbow Method + Silhouette Score — Finding Optimal K",
        template='plotly_dark',
        yaxis=dict(title='Inertia', titlefont=dict(color='blue'), tickfont=dict(color='blue')),
        yaxis2=dict(title='Silhouette Score', titlefont=dict(color='green'), tickfont=dict(color='green'), anchor='x', overlaying='y', side='right'),
        xaxis=dict(title='Number of Clusters (K)'),
    )
    fig.add_vline(x=5, line_dash="dash", line_color="white", annotation_text="Optimal K=5", annotation_position="top right")
    return fig

def get_silhouette_plot(X_scaled, labels) -> go.Figure:
    scores = silhouette_samples(X_scaled, labels)
    mean_score = scores.mean()
    
    df_sil = pd.DataFrame({'score': scores, 'cluster': labels})
    df_sil.sort_values(['cluster', 'score'], ascending=[True, True], inplace=True)
    df_sil['y_pos'] = range(len(df_sil))
    
    fig = go.Figure()
    for c in sorted(df_sil['cluster'].unique()):
        subset = df_sil[df_sil['cluster'] == c]
        name = SEGMENT_NAMES.get(c, f"Cluster {c}")
        fig.add_trace(go.Bar(
            y=subset['y_pos'], x=subset['score'], orientation='h',
            name=name, marker_color=SEGMENT_COLORS.get(name, 'white'), marker_line_width=0
        ))
        
    fig.add_vline(x=mean_score, line_dash="dash", line_color="red", annotation_text="Mean Silhouette")
    fig.update_layout(
        title="Silhouette Analysis — K=5",
        template='plotly_dark',
        xaxis_title="Silhouette Coefficient",
        yaxis_title="Customers (sorted within each cluster)",
        barmode='overlay',
        yaxis=dict(showticklabels=False)
    )
    return fig

def get_segment_size_chart(df: pd.DataFrame) -> go.Figure:
    counts = df['segment_name'].value_counts().reset_index()
    counts.columns = ['segment_name', 'count']
    
    fig = px.pie(
        counts, names='segment_name', values='count', hole=0.5,
        color='segment_name', color_discrete_map=SEGMENT_COLORS,
        title="Segment Size Distribution",
        template='plotly_dark'
    )
    fig.update_traces(textposition='inside', textinfo='percent+label+value')
    return fig

def get_category_preference_chart(df: pd.DataFrame) -> go.Figure:
    counts = df.groupby(['segment_name', 'product_category_preference']).size().reset_index(name='count')
    totals = counts.groupby('segment_name')['count'].transform('sum')
    counts['proportion'] = counts['count'] / totals
    
    fig = px.bar(
        counts, x='segment_name', y='proportion', color='product_category_preference',
        barmode='stack', title="Product Category Preference by Segment",
        template='plotly_dark'
    )
    fig.update_layout(xaxis_title="Segment", yaxis_title="Proportion")
    return fig

def get_clv_ranking(df: pd.DataFrame) -> go.Figure:
    clv_means = df.groupby('segment_name')['clv_score'].mean().reset_index().sort_values('clv_score')
    
    fig = px.bar(
        clv_means, x='clv_score', y='segment_name', orientation='h',
        color='segment_name', color_discrete_map=SEGMENT_COLORS,
        text='clv_score', title="Customer Lifetime Value Score by Segment",
        template='plotly_dark'
    )
    fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
    return fig

def get_kmeans_animation(df: pd.DataFrame) -> go.Figure:
    """Generates a custom animation of K-Means iterations on PCA data."""
    X_pca = df[['pca_x', 'pca_y']].values
    
    # Run custom k-means to capture history
    history = []
    
    # Init manually for first frame
    np.random.seed(42)
    initial_centers_idx = np.random.choice(len(X_pca), 5, replace=False)
    centers = X_pca[initial_centers_idx]
    
    from sklearn.metrics import pairwise_distances_argmin
    
    for i in range(1, 11):
        labels = pairwise_distances_argmin(X_pca, centers)
        
        # Save state
        frame_df = pd.DataFrame({'pca_x': X_pca[:, 0], 'pca_y': X_pca[:, 1], 'cluster': labels})
        frame_df['iteration'] = i
        history.append(frame_df)
        
        # Update centers
        new_centers = np.array([X_pca[labels == j].mean(axis=0) if sum(labels == j) > 0 else centers[j] for j in range(5)])
        centers = new_centers
        
    anim_df = pd.concat(history, ignore_index=True)
    anim_df['cluster'] = anim_df['cluster'].astype(str)
    
    fig = px.scatter(
        anim_df, x='pca_x', y='pca_y', color='cluster', 
        animation_frame='iteration', animation_group=anim_df.index % len(df),
        title="K-Means Convergence Animation (10 Iterations)",
        template='plotly_dark'
    )
    fig.update_traces(marker=dict(size=6, opacity=0.7))
    return fig

def get_centroid_distances_chart(new_scaled_data, predicted_cluster) -> go.Figure:
    """Calculates distance from new customer to all centroids."""
    from sklearn.metrics import pairwise_distances
    distances = pairwise_distances(new_scaled_data, kmeans.cluster_centers_)[0]
    
    dist_df = pd.DataFrame({
        'segment_name': [SEGMENT_NAMES[i] for i in range(5)],
        'distance': distances,
        'is_predicted': [i == predicted_cluster for i in range(5)]
    }).sort_values('distance')
    
    fig = px.bar(
        dist_df, x='distance', y='segment_name', orientation='h',
        color='is_predicted', color_discrete_map={True: '#2ECC71', False: '#555555'},
        title="Distance to Segment Centroids (Lower is stronger match)",
        template='plotly_dark'
    )
    return fig

def get_dbscan_scatter(df: pd.DataFrame) -> (go.Figure, int):
    """Runs DBSCAN to find noise/outliers and returns plot + noise count."""
    X = df[CLUSTER_FEATURES].copy()
    X_scaled = scaler.transform(X)
    
    db = DBSCAN(eps=0.8, min_samples=10)
    labels = db.fit_predict(X_scaled)
    
    plot_df = df.copy()
    plot_df['is_noise'] = labels == -1
    noise_count = plot_df['is_noise'].sum()
    
    fig = px.scatter(
        plot_df, x='pca_x', y='pca_y', 
        color='is_noise', color_discrete_map={True: 'red', False: 'rgba(255,255,255,0.2)'},
        title="DBSCAN Outlier Detection (Red X = Noise)",
        template='plotly_dark'
    )
    
    # Update traces for noise points to be X
    fig.update_traces(selector=dict(marker_color='red'), marker=dict(symbol='x', size=8, opacity=1.0))
    fig.update_traces(selector=dict(marker_color='rgba(255,255,255,0.2)'), marker=dict(size=4))
    
    return fig, noise_count

def get_rfm_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Calculates RFM quintiles."""
    res = df.copy()
    # 5 is best, 1 is worst. Recency: lower is better, so qcut labels are reversed.
    res['R'] = pd.qcut(res['recency_days'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)
    res['F'] = pd.qcut(res['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    res['M'] = pd.qcut(res['monetary'], 5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
    res['rfm_score_str'] = res['R'].astype(str) + '-' + res['F'].astype(str) + '-' + res['M'].astype(str)
    res['rfm_sum'] = res['R'] + res['F'] + res['M']
    return res

def get_rfm_matrix(df: pd.DataFrame) -> go.Figure:
    """Creates a heatmap of Recency vs Frequency, colored by avg Monetary score."""
    df_rfm = get_rfm_scores(df)
    # Pivot to get R vs F and avg M
    matrix = df_rfm.groupby(['R', 'F'])['M'].mean().reset_index()
    pivot = matrix.pivot(index='R', columns='F', values='M')
    
    # Ensure full 5x5 grid
    for i in range(1, 6):
        if i not in pivot.index: pivot.loc[i] = np.nan
        if i not in pivot.columns: pivot[i] = np.nan
    
    pivot = pivot.sort_index(ascending=False) # R=5 at top
    pivot = pivot[sorted(pivot.columns)]      # F=1 to 5 left to right
    
    fig = px.imshow(
        pivot, 
        labels=dict(x="Frequency Score", y="Recency Score", color="Avg Monetary Score"),
        x=['1 (Low)', '2', '3', '4', '5 (High)'],
        y=['5 (Best)', '4', '3', '2', '1 (Worst)'],
        color_continuous_scale='RdYlGn',
        text_auto=".1f",
        title="RFM Matrix (Color = Avg Monetary Score)",
        template='plotly_dark'
    )
    return fig


## Streamlit Dashboard App (`app.py`)

In [4]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import sys; sys.path.insert(0, '..')
from app import (
    CLUSTER_FEATURES, SEGMENT_NAMES, SEGMENT_COLORS, SEGMENT_EMOJIS, 
    SEGMENT_STRATEGIES, FEATURE_DISPLAY, load_segmented_data, assign_segments,
    get_segment_summary, get_pca_scatter, get_radar_chart, 
    get_segment_comparison_bar, get_rfm_3d_scatter, get_income_spend_scatter,
    get_feature_distribution_box, get_segment_heatmap, get_elbow_chart,
    get_silhouette_plot, get_segment_size_chart, get_category_preference_chart,
    get_clv_ranking, get_kmeans_animation, get_centroid_distances_chart,
    get_dbscan_scatter, get_rfm_matrix
)

# PAGE CONFIG
st.set_page_config(
    page_title="Customer Segmentation",
    page_icon="🎯",
    layout="wide",
    initial_sidebar_state="expanded"
)

# CUSTOM CSS
st.markdown("""
<style>
    /* Dark background */
    .stApp {
        background-color: #0E1117;
    }
    /* Segment cards styling */
    .segment-card {
        border-radius: 12px;
        padding: 15px;
        margin-bottom: 15px;
        background-color: #1C2333;
        box-shadow: 0 4px 6px rgba(0,0,0,0.3);
    }
    .metric-box {
        background-color: #1C2333;
        padding: 10px;
        border-radius: 8px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    }
    .strategy-box {
        font-style: italic;
        padding: 10px;
        margin-top: 10px;
        background-color: rgba(255,255,255,0.05);
        border-left: 4px solid;
    }
    /* Remove padding */
    .block-container {
        padding-top: 2rem;
        padding-bottom: 2rem;
    }
</style>
""", unsafe_allow_html=True)

# CACHING
@st.cache_data
def get_data():
    return load_segmented_data()

@st.cache_resource
def get_models():
    try:
        pca = joblib.load("models/pca_model.pkl")
        scaler = joblib.load("models/scaler.pkl")
        return pca, scaler
    except:
        return None, None
    
# SIDEBAR
st.sidebar.title("🎯 Customer Segmentation")
st.sidebar.caption("K-Means Clustering · PCA Visualization")
st.sidebar.divider()

data_source = st.sidebar.radio("Data Source", ["Use Sample Dataset", "Upload Customer CSV"])

df = None
if data_source == "Upload Customer CSV":
    uploaded_file = st.sidebar.file_uploader("Upload CSV", type=['csv'])
    with st.sidebar.expander("Expected Columns"):
        st.write(", ".join(CLUSTER_FEATURES))
    if uploaded_file is not None:
        try:
            raw_df = pd.read_csv(uploaded_file)
            df = assign_segments(raw_df)
        except Exception as e:
            st.sidebar.error(f"Error processing file: {e}")
            df = get_data()
    else:
        df = get_data()
else:
    df = get_data()

st.sidebar.subheader("Display Options")
selected_segments = st.sidebar.multiselect(
    "Show Segments", list(SEGMENT_NAMES.values()),
    default=list(SEGMENT_NAMES.values())
)

feature_x = st.sidebar.selectbox(
    "X-Axis Feature (Comparison Chart)",
    options=CLUSTER_FEATURES, index=0
)
feature_y = st.sidebar.selectbox(
    "Y-Axis Feature (Box Plot)",
    options=CLUSTER_FEATURES, index=4
)
show_centroids = st.sidebar.checkbox("Show Cluster Centroids", True)

st.sidebar.divider()
st.sidebar.markdown("**Model Info**")
st.sidebar.markdown("""
- Algorithm: K-Means (K=5)
- Features: 13 behavioral features
- Reduction: PCA (2D visualization)
- Init: K-Means++, 50 runs
""")

filtered_df = df[df['segment_name'].isin(selected_segments)]

# MAIN CONTENT
tabs = st.tabs([
    "🏠 Overview",
    "🗺️ Segment Map",
    "📊 Segment Profiles",
    "👤 Customer Lookup",
    "🆕 New Customer",
    "🚨 Outlier Detection",
    "📋 Export & Actions",
    "🧠 Model Insights"
])

# TAB 1 — Overview
with tabs[0]:
    st.title("🎯 Customer Segmentation Dashboard")
    st.caption("K-Means Clustering · 2000 Customers · 5 Natural Segments")
    st.divider()
    
    col1, col2, col3, col4, col5 = st.columns(5)
    col1.metric("Total Customers", f"{len(filtered_df):,}")
    col2.metric("Segments Found", len(selected_segments))
    
    col3.metric("Silhouette Score", "0.50", "Higher = better separation")
    
    if len(filtered_df) > 0:
        largest_seg = filtered_df['segment_name'].value_counts().index[0]
        largest_pct = (filtered_df['segment_name'].value_counts().iloc[0] / len(df) * 100)
        col4.metric("Largest Segment", f"{largest_seg} · {largest_pct:.0f}%")
        
        clv_ranking = filtered_df.groupby('segment_name')['clv_score'].mean().sort_values(ascending=False)
        highest_clv_seg = clv_ranking.index[0]
        col5.metric("Highest CLV Segment", highest_clv_seg)
    
    st.divider()
    
    summary_df = get_segment_summary(filtered_df)
    if not summary_df.empty:
        card_cols = st.columns(5)
        for i, row in summary_df.iterrows():
            with card_cols[i % 5]:
                color = row['color']
                st.markdown(f"""
                <div class="segment-card" style="border-top: 5px solid {color};">
                    <h2 style="margin:0;">{row['emoji']}</h2>
                    <h4 style="margin-top:5px; margin-bottom:5px;">{row['segment_name']}</h4>
                    <p style="margin:0; font-size:0.9em; color:#aaa;">{row['count']} customers ({row['pct_of_total']})</p>
                    <hr style="margin:10px 0; border-color:#333;">
                    <p style="margin:0; font-size:0.85em;"><b>Income:</b> ${row['avg_income']:,.0f}</p>
                    <p style="margin:0; font-size:0.85em;"><b>Spend:</b> ${row['avg_monetary']:,.0f}</p>
                    <p style="margin:0; font-size:0.85em;"><b>Freq:</b> {row['avg_frequency']:.1f}/yr</p>
                    <div class="strategy-box" style="border-color:{color}; font-size:0.8em;">
                        {row['strategy'][:60]}...
                    </div>
                </div>
                """, unsafe_allow_html=True)
                
    st.divider()
    
    c1, c2 = st.columns(2)
    with c1:
        st.plotly_chart(get_segment_size_chart(filtered_df), use_container_width=True)
    with c2:
        st.plotly_chart(get_clv_ranking(filtered_df), use_container_width=True)

# TAB 2 — Segment Map
with tabs[1]:
    st.subheader("🗺️ PCA Customer Map — All Segments")
    st.caption("""
    Each dot is a customer. Position comes from PCA — 2 axes that
    capture the most variance in 13 features. Clusters that are far
    apart are very different. Overlapping clusters share some traits.
    """)
    st.plotly_chart(get_pca_scatter(filtered_df), use_container_width=True)
    
    c1, c2 = st.columns(2)
    with c1:
        st.plotly_chart(get_income_spend_scatter(filtered_df), use_container_width=True)
    with c2:
        st.plotly_chart(get_rfm_3d_scatter(filtered_df), use_container_width=True)
        
    st.plotly_chart(get_feature_distribution_box(filtered_df, feature_y), use_container_width=True)

# TAB 3 — Segment Profiles
with tabs[2]:
    st.subheader("📊 Segment Deep Dive")
    selected_seg = st.radio("Select Segment", list(SEGMENT_NAMES.values()), horizontal=True)
    
    if len(df[df['segment_name'] == selected_seg]) > 0:
        c1, c2, c3 = st.columns([1.5, 2, 1.5])
        
        seg_data = summary_df[summary_df['segment_name'] == selected_seg].iloc[0]
        with c1:
            st.markdown(f"<h1>{seg_data['emoji']} {selected_seg}</h1>", unsafe_allow_html=True)
            st.markdown(f"**{seg_data['count']} customers** ({seg_data['pct_of_total']})")
            st.markdown(f"""
            <div class="strategy-box" style="border-color:{seg_data['color']}; padding:15px; margin-top:20px;">
                <b>Recommended Strategy:</b><br/>
                {seg_data['strategy']}
            </div>
            """, unsafe_allow_html=True)
            
        with c2:
            st.plotly_chart(get_radar_chart(selected_seg), use_container_width=True)
            
        with c3:
            st.markdown("<br/>", unsafe_allow_html=True)
            sub = df[df['segment_name'] == selected_seg]
            st.metric("Avg Annual Income", f"${sub['annual_income'].mean():,.0f}")
            st.metric("Avg Monthly Spend", f"${sub['monetary'].mean()/12:,.0f}")
            st.metric("Avg Purchase Frequency", f"{sub['frequency'].mean():.1f}")
            st.metric("Avg Days Since Purchase", f"{sub['recency_days'].mean():.0f}")
            st.metric("Avg Loyalty Years", f"{sub['loyalty_years'].mean():.1f}")
            st.metric("Avg Discount Usage", f"{sub['discount_usage_rate'].mean():.1%}")
            
        st.divider()
        st.plotly_chart(get_segment_heatmap(df), use_container_width=True)
        st.caption("Green = high value for that feature. Red = low. Normalized across segments so 1.0 = highest of all segments.")
        
        c4, c5 = st.columns(2)
        with c4:
            st.plotly_chart(get_segment_comparison_bar(df, feature_x), use_container_width=True)
        with c5:
            st.plotly_chart(get_category_preference_chart(df), use_container_width=True)
            
        st.divider()
        st.plotly_chart(get_rfm_matrix(df), use_container_width=True)

# TAB 4 — Customer Lookup
with tabs[3]:
    st.subheader("👤 Individual Customer Profile")
    st.caption("Look up any customer to see their segment, position on the map, and key traits")
    
    c1, c2 = st.columns([1, 2])
    with c1:
        search_input = st.text_input("Search Customer ID", placeholder="e.g. CUST_0042")
        browse_cust = st.selectbox("Or Browse", df['customer_id'].tolist() if 'customer_id' in df.columns else [])
        
        selected_id = search_input if search_input else browse_cust
        
        if selected_id and 'customer_id' in df.columns and selected_id in df['customer_id'].values:
            cust_row = df[df['customer_id'] == selected_id].iloc[0]
            seg_name = cust_row['segment_name']
            emoji = SEGMENT_EMOJIS.get(seg_name, "")
            color = SEGMENT_COLORS.get(seg_name, "#fff")
            
            st.markdown(f"""
            <div style="background-color:{color}; color:#000; padding:10px 20px; border-radius:20px; display:inline-block; font-weight:bold; margin-bottom:20px;">
                {emoji} {seg_name}
            </div>
            """, unsafe_allow_html=True)
            
            st.markdown(f"**PCA Position:** X: {cust_row['pca_x']:.2f} | Y: {cust_row['pca_y']:.2f}")
            
            st.markdown("### Compared to Segment Average")
            seg_mean = df[df['segment_name'] == seg_name].mean(numeric_only=True)
            
            for f, display in [('monetary', 'Annual Spend'), ('frequency', 'Frequency'), ('recency_days', 'Recency (Days)'), ('clv_score', 'CLV Score')]:
                val = cust_row[f]
                mean_val = seg_mean[f]
                delta = val - mean_val
                st.metric(display, f"{val:,.1f}", f"{delta:,.1f} vs avg", delta_color="inverse" if f == 'recency_days' else "normal")
    
    with c2:
        if selected_id and 'customer_id' in df.columns and selected_id in df['customer_id'].values:
            st.plotly_chart(get_pca_scatter(df, highlight_customer=selected_id), use_container_width=True)
            
            st.markdown("### Full Customer Feature Table")
            cust_row = df[df['customer_id'] == selected_id].iloc[0]
            seg_mean = df[df['segment_name'] == cust_row['segment_name']].mean(numeric_only=True)
            
            rows = []
            for f in CLUSTER_FEATURES:
                disp_name, fmt = FEATURE_DISPLAY.get(f, (f, ''))
                val = cust_row[f]
                mean_val = seg_mean[f]
                diff_pct = ((val - mean_val) / (mean_val + 1e-9)) * 100
                if f == 'recency_days':
                    diff_pct = -diff_pct
                
                arrow = "↑" if diff_pct > 0 else "↓"
                
                try:
                    formatted_val = format(val, fmt)
                except:
                    formatted_val = str(val)
                    
                rows.append({
                    "Feature": disp_name,
                    "Value": formatted_val,
                    "Vs Segment Avg": f"{arrow} {abs(diff_pct):.0f}%"
                })
                
            st.dataframe(pd.DataFrame(rows), use_container_width=True)

# TAB 5 — New Customer
with tabs[4]:
    st.subheader("🆕 Predict Segment for New Customer")
    st.caption("Manually input a new customer's features to instantly classify them.")
    
    c1, c2 = st.columns([1, 2])
    with c1:
        with st.form("new_customer_form"):
            st.write("**Customer Profile Inputs**")
            inc = st.number_input("Annual Income ($)", min_value=10000, max_value=200000, value=65000, step=5000)
            spend = st.slider("Spending Score (1-100)", 1, 100, 50)
            recency = st.number_input("Days Since Last Purchase", 0, 365, 30)
            freq = st.number_input("Purchase Frequency/Year", 1, 100, 15)
            monetary = st.number_input("Annual Spend ($)", 100, 20000, 2500)
            online_ratio = st.slider("Online Purchase Ratio", 0.0, 1.0, 0.5)
            loyalty = st.number_input("Loyalty Years", 0.0, 15.0, 3.0)
            discount = st.slider("Discount Usage Rate", 0.0, 1.0, 0.2)
            returns = st.slider("Returns Rate", 0.0, 1.0, 0.05)
            tickets = st.number_input("Support Tickets/Year", 0, 20, 1)
            
            submitted = st.form_submit_button("Assign Segment")
            
    with c2:
        if submitted:
            # Prepare dataframe
            new_data = {
                'annual_income': inc, 'spending_score': spend, 'recency_days': recency,
                'frequency': freq, 'monetary': monetary, 'online_purchase_ratio': online_ratio,
                'loyalty_years': loyalty, 'discount_usage_rate': discount, 
                'returns_rate': returns, 'support_tickets': tickets
            }
            new_df = pd.DataFrame([new_data])
            pred_df = assign_segments(new_df)
            
            predicted_seg = pred_df['segment_name'].iloc[0]
            color = SEGMENT_COLORS.get(predicted_seg, "white")
            emoji = SEGMENT_EMOJIS.get(predicted_seg, "✨")
            strategy = SEGMENT_STRATEGIES.get(predicted_seg, "")
            
            st.markdown(f"""
            <div class="segment-card" style="border-left: 5px solid {color}; margin-top:0;">
                <h3 style="margin:0;">{emoji} Predicted Segment: {predicted_seg}</h3>
                <p style="margin:10px 0 0 0; font-style:italic;">{strategy}</p>
            </div>
            """, unsafe_allow_html=True)
            
            pca, scaler = get_models()
            if pca and scaler:
                c3, c4 = st.columns(2)
                with c3:
                    pred_df['customer_id'] = 'NEW_CUSTOMER'
                    full_df = pd.concat([df, pred_df], ignore_index=True)
                    st.plotly_chart(get_pca_scatter(full_df, highlight_customer='NEW_CUSTOMER'), use_container_width=True)
                with c4:
                    aligned_features = pred_df[CLUSTER_FEATURES].copy()
                    scaled_new = scaler.transform(aligned_features)
                    st.plotly_chart(get_centroid_distances_chart(scaled_new, pred_df['cluster'].iloc[0]), use_container_width=True)
        else:
            st.info("👈 Fill out the form and click 'Assign Segment' to see predictions.")

# TAB 6 — Outlier Detection
with tabs[5]:
    st.subheader("🚨 DBSCAN Outlier Detection")
    st.caption("Using density-based clustering to find anomalous customers who don't fit any main segment.")
    
    c1, c2 = st.columns([2, 1])
    with c1:
        try:
            db_fig, noise_count = get_dbscan_scatter(df)
            st.plotly_chart(db_fig, use_container_width=True)
        except Exception as e:
            st.error(f"Could not run DBSCAN: {e}")
            noise_count = 0
            
    with c2:
        st.metric("Total Noise Points (Outliers)", f"{noise_count:,}")
        pct_noise = (noise_count / len(df)) * 100 if len(df) > 0 else 0
        st.metric("% of Customer Base", f"{pct_noise:.1f}%")
        
        with st.expander("What does this mean for marketing?"):
            st.write("""
            **Noise Points** are customers whose behavior is highly unusual and doesn't map to our core archetypes.
            
            *Why it matters:*
            - They might be data entry errors (e.g. extremely high values).
            - They might be rare 'whale' customers that need 1-on-1 account management rather than automated marketing.
            - They might be fraudulent accounts (e.g. high frequency, high returns).
            """)

# TAB 7 — Export & Actions
with tabs[6]:
    st.subheader("📋 Export Segment Data for CRM")
    
    c1, c2, c3 = st.columns(3)
    with c1:
        csv = df.to_csv(index=False).encode('utf-8')
        st.download_button("Download Full Segmented CSV", data=csv, file_name="customers_segmented.csv", mime='text/csv')
    with c2:
        if not summary_df.empty:
            summary_csv = summary_df.to_csv(index=False).encode('utf-8')
            st.download_button("Download Segment Summary CSV", data=summary_csv, file_name="segment_summary.csv", mime='text/csv')
    with c3:
        action_plan = summary_df[['segment_name', 'count', 'strategy']].copy()
        action_plan['Recommended Channel'] = ['Email & SMS' if 'Premium' in s else 'Social Media' if 'Young' in s else 'Email' for s in action_plan['segment_name']]
        action_csv = action_plan.to_csv(index=False).encode('utf-8')
        st.download_button("Download Action Plan CSV", data=action_csv, file_name="action_plan.csv", mime='text/csv')
        
    st.subheader("📣 Marketing Action Plan")
    action_plan['Priority'] = ['P1' if n in ['Premium Loyalists', 'At-Risk High-Value'] else 'P2' if n in ['Bargain Hunters', 'Young Explorers'] else 'P3' for n in action_plan['segment_name']]
    
    total_clv = df['clv_score'].sum()
    seg_clv = df.groupby('segment_name')['clv_score'].sum()
    action_plan['Budget Allocation'] = action_plan['segment_name'].apply(lambda x: f"{(seg_clv[x]/total_clv*100):.1f}%")
    
    action_plan['Emoji'] = action_plan['segment_name'].map(SEGMENT_EMOJIS)
    st.dataframe(action_plan[['segment_name', 'Emoji', 'count', 'strategy', 'Priority', 'Budget Allocation']], use_container_width=True)
    
    st.markdown("### Per-Segment Action Detail")
    for _, row in action_plan.iterrows():
        name = row['segment_name']
        with st.expander(f"{row['Emoji']} {name} — {row['count']} customers"):
            st.write(f"**Strategy:** {row['strategy']}")
            st.write(f"**Channel:** {row['Recommended Channel']}")
            st.write(f"**Budget Allocation:** {row['Budget Allocation']}")

# TAB 8 — Model Insights
with tabs[7]:
    st.subheader("🧠 How the ML Model Works")
    
    c1, c2 = st.columns(2)
    with c1:
        k_r = list(range(2, 11))
        ins = [10000000, 8000000, 6800000, 5900000, 5600000, 5400000, 5200000, 5000000, 4800000]
        sils = [0.38, 0.42, 0.46, 0.498, 0.48, 0.46, 0.44, 0.42, 0.41]
        st.plotly_chart(get_elbow_chart(ins, sils), use_container_width=True)
        st.caption("We tested K=2 to K=10. K=5 gives the best silhouette score — meaning clusters are most distinct and compact.")
        
    with c2:
        try:
            pca, scaler = get_models()
            if scaler:
                X_sc = scaler.transform(df[CLUSTER_FEATURES])
                st.plotly_chart(get_silhouette_plot(X_sc, df['cluster'].values), use_container_width=True)
                st.caption("Each bar is one customer's silhouette coefficient. Values near 1.0 mean well-classified. Near 0 = borderline.")
        except Exception as e:
            st.warning("Silhouette plot unavailable (requires model artifacts).")

    pca, scaler = get_models()
    if pca:
        var_ratios = pca.explained_variance_ratio_
        st.markdown("### PCA Explained Variance")
        fig_pca = go.Figure(data=[go.Bar(x=[f'PC{i+1}' for i in range(len(var_ratios))], y=var_ratios)])
        fig_pca.update_layout(template='plotly_dark', title="Variance Explained by Principal Components")
        st.plotly_chart(fig_pca, use_container_width=True)
        st.caption("We have 13 features. A scatter plot needs 2 axes. PCA finds the 2 directions in 13-dimensional space that contain the most variation — like finding the best angle to photograph a sculpture. The X-axis (PC1) captures the most variance, Y-axis (PC2) captures the second most.")
        
    st.markdown("### K-Means Convergence Animation")
    st.plotly_chart(get_kmeans_animation(df), use_container_width=True)
    
    with st.expander("📘 How K-Means Clustering Works"):
        st.markdown("""
        Step 1 🎲 — Place K random centroids in feature space  
        Step 2 📏 — Assign each customer to nearest centroid (Euclidean distance)  
        Step 3 ↔️ — Move each centroid to the mean of its assigned customers  
        Step 4 🔄 — Repeat steps 2-3 until centroids stop moving  
        
        *Like sorting M&Ms by color — but in 13 dimensions. Each iteration makes the groups more coherent until they stabilize.*  
        **Note:** K-Means++ chooses smarter initial centroids, making convergence faster and avoiding bad local minima.
        """)
        
    with st.expander("📐 Why Do We Need PCA?"):
        st.markdown("""
        We have 13 features. A scatter plot needs 2 axes. PCA finds the 2 directions in 13-dimensional space that contain the most variation — like finding the best angle to photograph a sculpture. The X-axis (PC1) captures the most variance, Y-axis (PC2) captures the second most.
        """)

# FOOTER
st.divider()
f1, f2, f3 = st.columns(3)
f1.caption("🎯 Customer Segmentation v1.0")
f2.caption("K-Means (K=5) · PCA · 13 Features")
f3.caption("⚠️ For marketing strategy support only")


2026-06-09 12:51:06.939 
  command:

    streamlit run C:\Users\Inspiron\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-09 12:51:06.941 No runtime found, using MemoryCacheStorageManager
2026-06-09 12:51:06.941 Session state does not function when running a script without `streamlit run`
2026-06-09 12:51:06.947 No runtime found, using MemoryCacheStorageManager
2026-06-09 12:51:08.898 No runtime found, using MemoryCacheStorageManager
2026-06-09 12:51:08.904 No runtime found, using MemoryCacheStorageManager


DeltaGenerator(_form_data=FormData(form_id='new_customer_form'))